In [ ]:
%pip install pandas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df=pd.read_csv("technical_debt_dataset_original.csv")

In [ ]:
df.head(10)

,projectname,classification,commenttext
0,apache-ant-1.7.0,DEFECT,// FIXME formatters are not thread-safe
1,apache-ant-1.7.0,DEFECT,"// XXX: (Jon Skeet) The comment ""if it hasn't ..."
2,apache-ant-1.7.0,DEFECT,"// I hate to admit it, but we don't know what ..."
3,apache-ant-1.7.0,DEFECT,// Just a note: StarTeam has a status for NEW ...
4,apache-ant-1.7.0,DEFECT,// the generated classes must not be added in ...
5,apache-ant-1.7.0,DEFECT,// todo: is this comment still relevant ?? // ...
6,apache-ant-1.7.0,DEFECT,// This is turned off temporarily. There appea...
7,apache-ant-1.7.0,DEFECT,// pattern now holds ** while string is not ex...
8,apache-ant-1.7.0,DEFECT,"// FIXME: Is ""No Namespace is Empty Namespace""..."
9,apache-ant-1.7.0,DEFECT,// FIXME this is actually not very cpu cycles ...


In [ ]:
print(df.describe())

       projectname          classification commenttext
count        62275                   62275       62275
unique          10                       6       38816
top      jEdit-4.2  WITHOUT_CLASSIFICATION       //}}}
freq         10322                   58204        3561


In [ ]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62275 entries, 0 to 62274
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   projectname     62275 non-null  object
 1   classification  62275 non-null  object
 2   commenttext     62275 non-null  object
dtypes: object(3)
memory usage: 1.4+ MB
None


In [ ]:
print(df.isnull().sum())

projectname       0
classification    0
commenttext       0
dtype: int64


In [ ]:
print(df["projectname"].unique())
print("unique count=",df["projectname"].nunique())

['apache-ant-1.7.0' 'apache-jmeter-2.10' 'argouml' 'columba-1.4-src'
 'emf-2.4.1' 'hibernate-distribution-3.3.2.GA' 'jEdit-4.2'
 'jfreechart-1.0.19' 'jruby-1.4.0' 'sql12']
unique count= 10


In [ ]:
df.duplicated()


,0
0,False
1,False
2,False
3,False
4,False
...,...
62270,False
62271,False
62272,False
62273,False


DATA SEPERATION


In [ ]:
import os
import zipfile
from google.colab import files

# 1. Create a directory to store the individual CSV files
output_dir = "project_csvs"
os.makedirs(output_dir, exist_ok=True)

# 2. Loop through each unique project name and save as a separate CSV
for project_name, group_df in df.groupby("projectname"):
  
  safe_name = str(project_name).replace("/", "_").replace("\\", "_")
  file_path = os.path.join(output_dir, f"{safe_name}.csv")

  
  group_df.to_csv(file_path, index=False)
  print(f"Created: {file_path}")

# Zip all the CSV files 
zip_filename = "all_projects.zip"
with zipfile.ZipFile(zip_filename, "w") as zipf:
  for root, dirs, files_in_dir in os.walk(output_dir):
    for file in files_in_dir:
      zipf.write(os.path.join(root, file), arcname=file)

print(f"\nSuccessfully zipped all files into: {zip_filename}")

# download of the zip file 
# files.download(zip_filename)

Created: project_csvs/apache-ant-1.7.0.csv
Created: project_csvs/apache-jmeter-2.10.csv
Created: project_csvs/argouml.csv
Created: project_csvs/columba-1.4-src.csv
Created: project_csvs/emf-2.4.1.csv
Created: project_csvs/hibernate-distribution-3.3.2.GA.csv
Created: project_csvs/jEdit-4.2.csv
Created: project_csvs/jfreechart-1.0.19.csv
Created: project_csvs/jruby-1.4.0.csv
Created: project_csvs/sql12.csv

Successfully zipped all files into: all_projects.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Create a dictionary containing all 10 DataFrames
dfs = {project: group.copy().reset_index(drop=True)
    for project, group in df.groupby("projectname")}

print("Available project keys:", list(dfs.keys()))
dfs["argouml"].columns
dfs["argouml"].info()
dfs["argouml"].isnull().sum()

Available project keys: ['apache-ant-1.7.0', 'apache-jmeter-2.10', 'argouml', 'columba-1.4-src', 'emf-2.4.1', 'hibernate-distribution-3.3.2.GA', 'jEdit-4.2', 'jfreechart-1.0.19', 'jruby-1.4.0', 'sql12']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9452 entries, 0 to 9451
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   projectname     9452 non-null   object
 1   classification  9452 non-null   object
 2   commenttext     9452 non-null   object
dtypes: object(3)
memory usage: 221.7+ KB


,0
projectname,0
classification,0
commenttext,0


Duplicate Value Handling


In [ ]:
for project, group_df in dfs.items():
  dupes = group_df.duplicated().sum()
  print(f"{project}: {dupes} duplicate rows")

apache-ant-1.7.0: 960 duplicate rows
apache-jmeter-2.10: 3802 duplicate rows
argouml: 3743 duplicate rows
columba-1.4-src: 2139 duplicate rows
emf-2.4.1: 1547 duplicate rows
hibernate-distribution-3.3.2.GA: 378 duplicate rows
jEdit-4.2: 5563 duplicate rows
jfreechart-1.0.19: 1779 duplicate rows
jruby-1.4.0: 1075 duplicate rows
sql12: 2252 duplicate rows


In [ ]:
total_size = 0
for project, group_df in dfs.items():
  before_count = len(group_df)
  dfs[project] = group_df.drop_duplicates().reset_index(drop=True)
  after_count = len(dfs[project])
  total_size += after_count
  print(
      f"{project}: Removed {before_count - after_count} duplicates (New total:"
      f" {after_count} rows)"
  )

print("total rows",total_size)

apache-ant-1.7.0: Removed 960 duplicates (New total: 3138 rows)
apache-jmeter-2.10: Removed 3802 duplicates (New total: 4255 rows)
argouml: Removed 3743 duplicates (New total: 5709 rows)
columba-1.4-src: Removed 2139 duplicates (New total: 4329 rows)
emf-2.4.1: Removed 1547 duplicates (New total: 2843 rows)
hibernate-distribution-3.3.2.GA: Removed 378 duplicates (New total: 2590 rows)
jEdit-4.2: Removed 5563 duplicates (New total: 4759 rows)
jfreechart-1.0.19: Removed 1779 duplicates (New total: 2629 rows)
jruby-1.4.0: Removed 1075 duplicates (New total: 3822 rows)
sql12: Removed 2252 duplicates (New total: 4963 rows)
total rows 39037


DownLoad Clean Dataframe

In [ ]:
import os
import zipfile
from google.colab import files


output_dir = "cleaned_project_csvs"
os.makedirs(output_dir, exist_ok=True)


for project_name, group_df in dfs.items():

  safe_name = str(project_name).replace("/", "_").replace("\\", "_")
  file_path = os.path.join(output_dir, f"{safe_name}_cleaned.csv")

  
  group_df.to_csv(file_path, index=False)
  print(f"Saved: {file_path}")

zip_filename = "cleaned_projects.zip"
with zipfile.ZipFile(zip_filename, "w") as zipf:
  for root, dirs, filenames in os.walk(output_dir):
    for filename in filenames:
      zipf.write(os.path.join(root, filename), arcname=filename)

print(f"\nAll files successfully zipped into {zip_filename}!")

# files.download(zip_filename)

Saved: cleaned_project_csvs/apache-ant-1.7.0_cleaned.csv
Saved: cleaned_project_csvs/apache-jmeter-2.10_cleaned.csv
Saved: cleaned_project_csvs/argouml_cleaned.csv
Saved: cleaned_project_csvs/columba-1.4-src_cleaned.csv
Saved: cleaned_project_csvs/emf-2.4.1_cleaned.csv
Saved: cleaned_project_csvs/hibernate-distribution-3.3.2.GA_cleaned.csv
Saved: cleaned_project_csvs/jEdit-4.2_cleaned.csv
Saved: cleaned_project_csvs/jfreechart-1.0.19_cleaned.csv
Saved: cleaned_project_csvs/jruby-1.4.0_cleaned.csv
Saved: cleaned_project_csvs/sql12_cleaned.csv

All files successfully zipped into cleaned_projects.zip!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Target Labeling


In [ ]:
for project_name,group_dfs in dfs.items():
  dfs[project_name]["target"]=dfs[project_name]["classification"].apply(
      lambda t:0 if str(t).upper()=="WITHOUT_CLASSIFICATION" else 1
      )
  print(project_name,"\n",dfs[project_name].head(2))

apache-ant-1.7.0 
         projectname classification  \
0  apache-ant-1.7.0         DEFECT   
1  apache-ant-1.7.0         DEFECT   

                                         commenttext  target  
0            // FIXME formatters are not thread-safe       1  
1  // XXX: (Jon Skeet) The comment "if it hasn't ...       1  
apache-jmeter-2.10 
           projectname classification  \
0  apache-jmeter-2.10         DEFECT   
1  apache-jmeter-2.10         DEFECT   

                                         commenttext  target  
0       // Don't try restoring the URL TODO: wy not?       1  
1  // TODO: Why do we clone the return value? Thi...       1  
argouml 
   projectname classification  \
0     argouml         DEFECT   
1     argouml         DEFECT   

                                         commenttext  target  
0  // TODO: This brute force approach of updating...       1  
1  /* TODO: The above statement also refreshes wh...       1  
columba-1.4-src 
        projectname classificatio

In [ ]:
from datetime import datetime
import os
import zipfile
from google.colab import files


def export_and_download_dfs(
    dfs_dict,
    base_dir="cleaned_project_csvs",
    base_zip_name="cleaned_projects",
):
    """Takes a dictionary of DataFrames, saves each as a CSV in a timestamped

    folder, zips them together with an ISO timestamp, and triggers a download.
    """
    # 1. Generate standard ISO timestamp ( 2026-10-19_16-29-00)
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

    output_dir = f"{base_dir}_{timestamp}"
    zip_filename = f"{base_zip_name}_{timestamp}.zip"

    os.makedirs(output_dir, exist_ok=True)

    
    for project_name, group_df in dfs_dict.items():
        safe_name = str(project_name).replace("/", "_").replace("\\", "_")
        file_path = os.path.join(output_dir, f"{safe_name}_cleaned.csv")

        group_df.to_csv(file_path, index=False)
        print(f"Saved: {file_path}")

    with zipfile.ZipFile(zip_filename, "w") as zipf:
        for root, dirs, filenames in os.walk(output_dir):
            for filename in filenames:
                zipf.write(os.path.join(root, filename), arcname=filename)

    print(f"\nAll files successfully zipped into {zip_filename}!")


    files.download(zip_filename)

In [ ]:
# export_and_download_dfs(dfs)

Project Attribute Display

In [ ]:
def printInfo():
  for project_name, group_df in dfs.items():
    print(f"=== Project: {project_name} ===")
    group_df.info()
    print("\n" + "=" * 40 + "\n")
printInfo()

=== Project: apache-ant-1.7.0 ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3138 entries, 0 to 3137
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   projectname     3138 non-null   object
 1   classification  3138 non-null   object
 2   commenttext     3138 non-null   object
 3   target          3138 non-null   int64 
dtypes: int64(1), object(3)
memory usage: 98.2+ KB


=== Project: apache-jmeter-2.10 ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4255 entries, 0 to 4254
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   projectname     4255 non-null   object
 1   classification  4255 non-null   object
 2   commenttext     4255 non-null   object
 3   target          4255 non-null   int64 
dtypes: int64(1), object(3)
memory usage: 133.1+ KB


=== Project: argouml ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5709 ent

**Preprocessing Steps**


In [ ]:
for project_name,group_dfs in dfs.items():
  group_dfs.drop(columns=["classification","projectname"],inplace=True,errors="ignore")

Droping Unnessary Columns


In [ ]:
printInfo()

=== Project: apache-ant-1.7.0 ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3138 entries, 0 to 3137
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   commenttext  3138 non-null   object
 1   target       3138 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 49.2+ KB


=== Project: apache-jmeter-2.10 ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4255 entries, 0 to 4254
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   commenttext  4255 non-null   object
 1   target       4255 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 66.6+ KB


=== Project: argouml ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5709 entries, 0 to 5708
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   commenttext  5709 non-null   object
 1   target       5709 non-nul

In [ ]:

dfs['apache-ant-1.7.0'].sample(6)

,commenttext,target
2581,// Exact match,0
3090,//execute the get,0
2243,// make sure we do not get a NPE,0
1845,// Pre-fetch the item properties and cache them.,0
469,// file doesn't exist -> property 'test' == null,0
11,"// TODO: // A warning line, that shows code, w...",1


In [ ]:
dfs.keys()

dict_keys(['apache-ant-1.7.0', 'apache-jmeter-2.10', 'argouml', 'columba-1.4-src', 'emf-2.4.1', 'hibernate-distribution-3.3.2.GA', 'jEdit-4.2', 'jfreechart-1.0.19', 'jruby-1.4.0', 'sql12'])

In [ ]:
printInfo()

=== Project: apache-ant-1.7.0 ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3138 entries, 0 to 3137
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   commenttext  3138 non-null   object
 1   target       3138 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 49.2+ KB


=== Project: apache-jmeter-2.10 ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4255 entries, 0 to 4254
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   commenttext  4255 non-null   object
 1   target       4255 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 66.6+ KB


=== Project: argouml ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5709 entries, 0 to 5708
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   commenttext  5709 non-null   object
 1   target       5709 non-nul

Model Training Pipeline

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectPercentile, mutual_info_classif
from sklearn.metrics import precision_recall_fscore_support
from sklearn.naive_bayes import MultinomialNB

In [ ]:
def run_cross_project_validation(dfs: dict, ratio: float = 0.1) -> dict:
  """Runs the cross-project Leave-One-Project-Out validation experiment

  with ensemble majority voting and Information Gain feature selection.

  Args:
      dfs (dict): Dictionary mapping project names to their pandas DataFrames.
      ratio (float): Fraction of top features to select via Information Gain
        (e.g., 0.1 for 10%).

  Returns:
      dict: A dictionary storing evaluation metrics for each target project.
  """
  experiment_results = {}

  for target_name, test_df in dfs.items():
    print(f"\n=== Target Project: {target_name} ===")
    y_test = test_df['target'].values.astype(int)

    # Initialize ballot box/vote array for this target project's test instances
    votes = np.zeros(len(test_df))

    
    for source_name, train_df in dfs.items():
      if source_name == target_name:
        continue

      # 1. Text Vectorization (TF-IDF)
      vectorizer = TfidfVectorizer()
      X_train = vectorizer.fit_transform(train_df['commenttext'])
      X_test = vectorizer.transform(test_df['commenttext'])
      y_train = train_df['target'].values.astype(int)

      # 2. Information Gain Feature Selection 
      percentile_val = int(ratio * 100)
      selector = SelectPercentile(
          score_func=mutual_info_classif, percentile=percentile_val
      )
      X_train_selected = selector.fit_transform(X_train, y_train)
      X_test_selected = selector.transform(X_test)

      # 3. Model Training (Naive Bayes Multinomial)
      clf = MultinomialNB()
      clf.fit(X_train_selected, y_train)

      # 4. Voting (+1 for positive SATD, -1 for negative)
      preds = clf.predict(X_test_selected)
      source_votes = np.where(preds == 1, 1, -1)
      votes += source_votes

    # 5. Aggregate Ensemble Votes and Evaluate
    final_preds = np.where(votes > 0, 1, 0)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, final_preds, average='binary', zero_division=0
    )

    experiment_results[target_name] = {
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }

    print(
        f"Final Metrics -> Precision: {precision:.4f} | Recall: {recall:.4f} |"
        f" F1-Score: {f1:.4f}"
    )

  return experiment_results

In [ ]:
dfs.keys()

dict_keys(['apache-ant-1.7.0', 'apache-jmeter-2.10', 'argouml', 'columba-1.4-src', 'emf-2.4.1', 'hibernate-distribution-3.3.2.GA', 'jEdit-4.2', 'jfreechart-1.0.19', 'jruby-1.4.0', 'sql12'])

In [ ]:
dfs.pop('apache-ant-1.7.0', None)
dfs.pop('emf-2.4.1', None)
dfs.keys()

dict_keys(['apache-jmeter-2.10', 'argouml', 'columba-1.4-src', 'hibernate-distribution-3.3.2.GA', 'jEdit-4.2', 'jfreechart-1.0.19', 'jruby-1.4.0', 'sql12'])

In [ ]:
result={}
result["using 10%"]=run_cross_project_validation(dfs,ratio=0.1)

In [ ]:
result["using 10%"]

{'apache-jmeter-2.10': {'precision': 0.9166666666666666,
  'recall': 0.03459119496855346,
  'f1': 0.06666666666666667},
 'argouml': {'precision': 0.5714285714285714,
  'recall': 0.017391304347826087,
  'f1': 0.03375527426160337},
 'columba-1.4-src': {'precision': 0.8461538461538461,
  'recall': 0.06748466257668712,
  'f1': 0.125},
 'hibernate-distribution-3.3.2.GA': {'precision': 0.6842105263157895,
  'recall': 0.030373831775700934,
  'f1': 0.058165548098434},
 'jEdit-4.2': {'precision': 1.0,
  'recall': 0.00851063829787234,
  'f1': 0.016877637130801686},
 'jfreechart-1.0.19': {'precision': 0.6666666666666666,
  'recall': 0.03669724770642202,
  'f1': 0.06956521739130435},
 'jruby-1.4.0': {'precision': 0.4444444444444444,
  'recall': 0.008658008658008658,
  'f1': 0.016985138004246284},
 'sql12': {'precision': 0.8571428571428571,
  'recall': 0.02631578947368421,
  'f1': 0.05106382978723404}}

In [ ]:
def run_cross_project_validation2(dfs: dict, ratio: float = 0.1) -> dict:
  """Runs the cross-project Leave-One-Project-Out validation experiment

  with ensemble majority voting and Information Gain feature selection.

  Args:
      dfs (dict): Dictionary mapping project names to their pandas DataFrames.
      ratio (float): Fraction of top features to select via Information Gain
        (e.g., 0.1 for 10%).

  Returns:
      dict: A dictionary storing evaluation metrics for each target project.
  """
  experiment_results = {}

  for target_name, test_df in dfs.items():
    print(f"\n=== Target Project: {target_name} ===")
    y_test = test_df['target'].values.astype(int)

    
    votes = np.zeros(len(test_df))

  
    for source_name, train_df in dfs.items():
      if source_name == target_name:
        continue

   
      vectorizer = TfidfVectorizer()
      X_train = vectorizer.fit_transform(train_df['commenttext'])
      X_test = vectorizer.transform(test_df['commenttext'])
      y_train = train_df['target'].values.astype(int)

      
      percentile_val = int(ratio * 100)
      selector = SelectPercentile(
          score_func=mutual_info_classif, percentile=percentile_val
      )
      X_train_selected = selector.fit_transform(X_train, y_train)
      X_test_selected = selector.transform(X_test)

      
      clf = MultinomialNB()
      clf.fit(X_train_selected, y_train)

      
      preds = clf.predict(X_test_selected)
      source_votes = np.where(preds == 1, 1, -1)
      votes += source_votes

    # 5. here vote>=0
    final_preds = np.where(votes >= 0, 1, 0)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, final_preds, average='binary', zero_division=0
    )

    experiment_results[target_name] = {
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }

    print(
        f"Final Metrics -> Precision: {precision:.4f} | Recall: {recall:.4f} |"
        f" F1-Score: {f1:.4f}"
    )

  return experiment_results

In [ ]:
result_using_10_percent=run_cross_project_validation2(dfs,ratio=0.1)

In [ ]:
result_using_10_percent

{'apache-jmeter-2.10': {'precision': 0.9166666666666666,
  'recall': 0.03459119496855346,
  'f1': 0.06666666666666667},
 'argouml': {'precision': 0.5714285714285714,
  'recall': 0.017391304347826087,
  'f1': 0.03375527426160337},
 'columba-1.4-src': {'precision': 0.8461538461538461,
  'recall': 0.06748466257668712,
  'f1': 0.125},
 'hibernate-distribution-3.3.2.GA': {'precision': 0.6842105263157895,
  'recall': 0.030373831775700934,
  'f1': 0.058165548098434},
 'jEdit-4.2': {'precision': 1.0,
  'recall': 0.00851063829787234,
  'f1': 0.016877637130801686},
 'jfreechart-1.0.19': {'precision': 0.6666666666666666,
  'recall': 0.03669724770642202,
  'f1': 0.06956521739130435},
 'jruby-1.4.0': {'precision': 0.4444444444444444,
  'recall': 0.008658008658008658,
  'f1': 0.016985138004246284},
 'sql12': {'precision': 0.8571428571428571,
  'recall': 0.02631578947368421,
  'f1': 0.05106382978723404}}

In [ ]:
import re
import warnings
import numpy as np
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectPercentile, mutual_info_classif
from sklearn.metrics import precision_recall_fscore_support
from sklearn.naive_bayes import MultinomialNB

In [ ]:



def run_cross_project_validation3(dfs: dict, ratio: float = 0.1) -> dict:
  """Runs the cross-project Leave-One-Project-Out validation experiment

  with ensemble majority voting, Porter stemming, and Information Gain feature
  selection.

  Args:
      dfs (dict): Dictionary mapping project names to their pandas DataFrames.
      ratio (float): Fraction of top features to select via Information Gain
        (e.g., 0.1 for 10%).

  Returns:
      dict: A dictionary storing evaluation metrics for each target project.
  """
  experiment_results = {}
  stemmer = PorterStemmer()

  # Define custom tokenizer to apply Porter stemming during vectorization
  def stemmed_tokenizer(doc):
    tokens = re.findall(r'(?u)\b\w\w+\b', doc.lower())
    return [stemmer.stem(t) for t in tokens]

  # Suppress internal sklearn warnings cleanly
  with warnings.catch_warnings():
    warnings.simplefilter('ignore')

    for target_name, test_df in dfs.items():
      print(f'\n=== Target Project: {target_name} ===')
      y_test = test_df['target'].values.astype(int)

      
      votes = np.zeros(len(test_df))

      
      for source_name, train_df in dfs.items():
        if source_name == target_name:
          continue

      
        vectorizer = TfidfVectorizer(
            tokenizer=stemmed_tokenizer, token_pattern=None
        )
        X_train = vectorizer.fit_transform(train_df['commenttext'])
        X_test = vectorizer.transform(test_df['commenttext'])
        y_train = train_df['target'].values.astype(int)

        
        percentile_val = int(ratio * 100)
        selector = SelectPercentile(
            score_func=mutual_info_classif, percentile=percentile_val
        )
        X_train_selected = selector.fit_transform(X_train, y_train)
        X_test_selected = selector.transform(X_test)

       
        clf = MultinomialNB()
        clf.fit(X_train_selected, y_train)

        
        preds = clf.predict(X_test_selected)
        source_votes = np.where(preds == 1, 1, -1)
        votes += source_votes

      # (using votes >= 0 threshold)
      final_preds = np.where(votes >= 0, 1, 0)

      precision, recall, f1, _ = precision_recall_fscore_support(
          y_test, final_preds, average='binary', zero_division=0
      )

      experiment_results[target_name] = {
          'precision': precision,
          'recall': recall,
          'f1': f1,
      }

      print(
          f'Final Metrics -> Precision: {precision:.4f} | Recall:'
          f' {recall:.4f} | F1-Score: {f1:.4f}'
      )

  return experiment_results

In [ ]:
result_using_10_percent_stemming=run_cross_project_validation3(dfs,ratio=0.1)


=== Target Project: apache-jmeter-2.10 ===
Final Metrics -> Precision: 0.9167 | Recall: 0.0346 | F1-Score: 0.0667

=== Target Project: argouml ===
Final Metrics -> Precision: 0.6286 | Recall: 0.0191 | F1-Score: 0.0371

=== Target Project: columba-1.4-src ===
Final Metrics -> Precision: 0.8182 | Recall: 0.0552 | F1-Score: 0.1034

=== Target Project: hibernate-distribution-3.3.2.GA ===
Final Metrics -> Precision: 0.6667 | Recall: 0.0280 | F1-Score: 0.0538

=== Target Project: jEdit-4.2 ===
Final Metrics -> Precision: 1.0000 | Recall: 0.0085 | F1-Score: 0.0169

=== Target Project: jfreechart-1.0.19 ===
Final Metrics -> Precision: 0.6667 | Recall: 0.0367 | F1-Score: 0.0696

=== Target Project: jruby-1.4.0 ===
Final Metrics -> Precision: 0.5833 | Recall: 0.0152 | F1-Score: 0.0295

=== Target Project: sql12 ===
Final Metrics -> Precision: 0.8750 | Recall: 0.0307 | F1-Score: 0.0593


In [ ]:
for project_name, df in dfs.items():
  # Get counts of 0s and 1s
  counts = df['target'].value_counts()
  total = len(df)

  negatives = counts.get(0, 0)
  positives = counts.get(1, 0)
  pos_percentage = (positives / total) * 100

  print(
      f"Project: {project_name} | Total: {total} | Clean (0): {negatives} |"
      f" SATD (1): {positives} ({pos_percentage:.2f}%)"
  )

Project: apache-jmeter-2.10 | Total: 4255 | Clean (0): 3937 | SATD (1): 318 (7.47%)
Project: argouml | Total: 5709 | Clean (0): 4559 | SATD (1): 1150 (20.14%)
Project: columba-1.4-src | Total: 4329 | Clean (0): 4166 | SATD (1): 163 (3.77%)
Project: hibernate-distribution-3.3.2.GA | Total: 2590 | Clean (0): 2162 | SATD (1): 428 (16.53%)
Project: jEdit-4.2 | Total: 4759 | Clean (0): 4524 | SATD (1): 235 (4.94%)
Project: jfreechart-1.0.19 | Total: 2629 | Clean (0): 2520 | SATD (1): 109 (4.15%)
Project: jruby-1.4.0 | Total: 3822 | Clean (0): 3360 | SATD (1): 462 (12.09%)
Project: sql12 | Total: 4963 | Clean (0): 4735 | SATD (1): 228 (4.59%)


In [ ]:
import re
import warnings
import numpy as np
from imblearn.over_sampling import SMOTE
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectPercentile, mutual_info_classif
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.naive_bayes import MultinomialNB


In [ ]:
def run_cross_project_validation_with_smote(dfs: dict, ratio: float = 0.1) -> dict:
  """Runs the cross-project Leave-One-Project-Out validation experiment

  with ensemble majority voting, Porter stemming, SMOTE over-sampling,
  and evaluates Precision, Recall, F1-Score, and Accuracy.

  Args:
      dfs (dict): Dictionary mapping project names to their pandas DataFrames.
      ratio (float): Fraction of top features to select via Information Gain
        (e.g., 0.1 for 10%).

  Returns:
      dict: A dictionary storing evaluation metrics for each target project.
  """
  experiment_results = {}
  stemmer = PorterStemmer()

  
  def stemmed_tokenizer(doc):
    tokens = re.findall(r'(?u)\b\w\w+\b', doc.lower())
    return [stemmer.stem(t) for t in tokens]

  with warnings.catch_warnings():
    warnings.simplefilter('ignore')

    for target_name, test_df in dfs.items():
      print(f'\n=== Target Project (SMOTE + Stemming): {target_name} ===')
      y_test = test_df['target'].values.astype(int)

      
      votes = np.zeros(len(test_df))

      for source_name, train_df in dfs.items():
        if source_name == target_name:
          continue

        
        vectorizer = TfidfVectorizer(
            tokenizer=stemmed_tokenizer, token_pattern=None
        )
        X_train = vectorizer.fit_transform(train_df['commenttext'])
        X_test = vectorizer.transform(test_df['commenttext'])
        y_train = train_df['target'].values.astype(int)

        
        percentile_val = int(ratio * 100)
        selector = SelectPercentile(
            score_func=mutual_info_classif, percentile=percentile_val
        )
        X_train_selected = selector.fit_transform(X_train, y_train)
        X_test_selected = selector.transform(X_test)

        # Apply SMOTE to address class imbalance on training data only
        smote = SMOTE(random_state=42)
        X_train_resampled, y_train_resampled = smote.fit_resample(
            X_train_selected, y_train
        )

        # Model Training (Naive Bayes Multinomial)
        clf = MultinomialNB()
        clf.fit(X_train_resampled, y_train_resampled)

        
        preds = clf.predict(X_test_selected)
        source_votes = np.where(preds == 1, 1, -1)
        votes += source_votes

      #  Aggregate Ensemble Votes and Evaluate
      final_preds = np.where(votes >= 0, 1, 0)

      precision, recall, f1, _ = precision_recall_fscore_support(
          y_test, final_preds, average='binary', zero_division=0
      )
      accuracy = accuracy_score(y_test, final_preds)

      experiment_results[target_name] = {
          'precision': precision,
          'recall': recall,
          'f1': f1,
          'accuracy': accuracy,
      }

      print(
          f'Final Metrics -> Accuracy: {accuracy:.4f} | Precision:'
          f' {precision:.4f} | Recall: {recall:.4f} | F1-Score: {f1:.4f}'
      )

  return experiment_results

In [ ]:
result_using_10_percent_stemming_smote=run_cross_project_validation_with_smote(dfs,ratio=0.1)


=== Target Project (SMOTE + Stemming): apache-jmeter-2.10 ===
Final Metrics -> Accuracy: 0.8338 | Precision: 0.2972 | Recall: 0.8962 | F1-Score: 0.4464

=== Target Project (SMOTE + Stemming): argouml ===
Final Metrics -> Accuracy: 0.8520 | Precision: 0.5919 | Recall: 0.8539 | F1-Score: 0.6992

=== Target Project (SMOTE + Stemming): columba-1.4-src ===
Final Metrics -> Accuracy: 0.9030 | Precision: 0.2676 | Recall: 0.9080 | F1-Score: 0.4134

=== Target Project (SMOTE + Stemming): hibernate-distribution-3.3.2.GA ===
Final Metrics -> Accuracy: 0.8112 | Precision: 0.4623 | Recall: 0.8738 | F1-Score: 0.6047

=== Target Project (SMOTE + Stemming): jEdit-4.2 ===
Final Metrics -> Accuracy: 0.5938 | Precision: 0.0801 | Recall: 0.6894 | F1-Score: 0.1436

=== Target Project (SMOTE + Stemming): jfreechart-1.0.19 ===
Final Metrics -> Accuracy: 0.8775 | Precision: 0.2396 | Recall: 0.8991 | F1-Score: 0.3784

=== Target Project (SMOTE + Stemming): jruby-1.4.0 ===
Final Metrics -> Accuracy: 0.8511 | P

In [ ]:


def run_cross_project_validation_flexible(
    dfs: dict,
    ratio: float = 0.1,
    vote_threshold: int = 0,
    smote_strategy: float = 1.0,
    classifier=None,
) -> dict:
  """Runs the cross-project Leave-One-Project-Out validation experiment

  with full hyperparameter tuning for feature ratio, ensemble voting
  threshold,
  SMOTE sampling strategy, and pluggable classifiers.

  Args:
      dfs (dict): Dictionary mapping project names to their pandas DataFrames.
      ratio (float): Fraction of top features to select via Information Gain
        (e.g., 0.1 for 10%).
      vote_threshold (int): Minimum net votes required to classify an instance
        as positive (e.g., 0 for simple majority, or higher like 2 or 4 for
        stricter consensus).
      smote_strategy (float): The sampling strategy for SMOTE (e.g., 1.0 for
        100% balance, 0.5 for partial balance).
      classifier: Instantiated scikit-learn classifier (defaults to MultinomialNB
        if None).

  Returns:
      dict: A dictionary storing evaluation metrics for each target project.
  """
  experiment_results = {}
  stemmer = PorterStemmer()


  if classifier is None:
    classifier = MultinomialNB()

  
  def stemmed_tokenizer(doc):
    tokens = re.findall(r'(?u)\b\w\w+\b', doc.lower())
    return [stemmer.stem(t) for t in tokens]

  with warnings.catch_warnings():
    warnings.simplefilter('ignore')

    for target_name, test_df in dfs.items():
      print(f'\n=== Target Project: {target_name} ===')
      y_test = test_df['target'].values.astype(int)

      
      votes = np.zeros(len(test_df))

      for source_name, train_df in dfs.items():
        if source_name == target_name:
          continue

        
        vectorizer = TfidfVectorizer(
            tokenizer=stemmed_tokenizer, token_pattern=None
        )
        X_train = vectorizer.fit_transform(train_df['commenttext'])
        X_test = vectorizer.transform(test_df['commenttext'])
        y_train = train_df['target'].values.astype(int)

        
        percentile_val = int(ratio * 100)
        selector = SelectPercentile(
            score_func=mutual_info_classif, percentile=percentile_val
        )
        X_train_selected = selector.fit_transform(X_train, y_train)
        X_test_selected = selector.transform(X_test)

        
        smote = SMOTE(sampling_strategy=smote_strategy, random_state=42)
        X_train_resampled, y_train_resampled = smote.fit_resample(
            X_train_selected, y_train
        )

        
        from sklearn.base import clone

        clf = clone(classifier)
        clf.fit(X_train_resampled, y_train_resampled)

        
        preds = clf.predict(X_test_selected)
        source_votes = np.where(preds == 1, 1, -1)
        votes += source_votes

      
      final_preds = np.where(votes >= vote_threshold, 1, 0)

      precision, recall, f1, _ = precision_recall_fscore_support(
          y_test, final_preds, average='binary', zero_division=0
      )
      accuracy = accuracy_score(y_test, final_preds)

      experiment_results[target_name] = {
          'precision': precision,
          'recall': recall,
          'f1': f1,
          'accuracy': accuracy,
      }

      print(
          f'Metrics -> Accuracy: {accuracy:.4f} | Precision: {precision:.4f} |'
          f' Recall: {recall:.4f} | F1-Score: {f1:.4f}'
      )

  return experiment_results

In [ ]:
from joblib import Parallel, delayed

In [ ]:


# Defining hyperparameter grids
smote_strategies = [0.5, 0.75, 1.0]
vote_thresholds = [0, 2, 4]
fixed_ratio = 0.1

# Flatten the grid into a list of individual tasks/configurations
grid_tasks = []
for smote_val in smote_strategies:
  for v_thresh in vote_thresholds:
    config_key = f'smote_{smote_val}_vote_{v_thresh}_ratio_{fixed_ratio}'
    grid_tasks.append((config_key, smote_val, v_thresh, fixed_ratio))


# Define a worker function to execute a single configuration
def run_single_config(config_key, smote_val, v_thresh, ratio):
  print(
      f'--- Starting: SMOTE={smote_val} | Vote Threshold={v_thresh} ---'
  )
  result = run_cross_project_validation_flexible(
      dfs=dfs,
      ratio=ratio,
      vote_threshold=v_thresh,
      smote_strategy=smote_val,
      classifier=None,
  )
  print(f'--- Finished: SMOTE={smote_val} | Vote Threshold={v_thresh} ---')
  return config_key, result


# Execute in parallel using joblib
# n_jobs=-1 uses all available CPU cores in your Colab environment
print(
    f'Starting parallel grid search across {len(grid_tasks)} configurations...'
)
parallel_results = Parallel(n_jobs=-1, verbose=10)(
    delayed(run_single_config)(key, s_val, v_val, r_val)
    for key, s_val, v_val, r_val in grid_tasks
)


grid_search_results = dict(parallel_results)
print(
    '\nParallel grid search completed! All results stored in'
    ' `grid_search_results`.'
)

Starting parallel grid search across 9 configurations...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed: 16.3min
[Parallel(n_jobs=-1)]: Done   4 tasks      | elapsed: 31.9min
[Parallel(n_jobs=-1)]: Done   7 out of   9 | elapsed: 62.3min remaining: 17.8min



Parallel grid search completed! All results stored in `grid_search_results`.


[Parallel(n_jobs=-1)]: Done   9 out of   9 | elapsed: 71.4min finished


In [ ]:
grid_search_results

{'smote_0.5_vote_0_ratio_0.1': {'apache-jmeter-2.10': {'precision': 0.6933333333333334,
   'recall': 0.6540880503144654,
   'f1': 0.6731391585760518,
   'accuracy': 0.9525264394829612},
  'argouml': {'precision': 0.7869034406215316,
   'recall': 0.6165217391304347,
   'f1': 0.6913700633837152,
   'accuracy': 0.8891224382553863},
  'columba-1.4-src': {'precision': 0.71875,
   'recall': 0.7055214723926381,
   'f1': 0.7120743034055728,
   'accuracy': 0.9785169785169785},
  'hibernate-distribution-3.3.2.GA': {'precision': 0.8244047619047619,
   'recall': 0.647196261682243,
   'f1': 0.725130890052356,
   'accuracy': 0.918918918918919},
  'jEdit-4.2': {'precision': 0.5957446808510638,
   'recall': 0.23829787234042554,
   'f1': 0.3404255319148936,
   'accuracy': 0.9544021853330532},
  'jfreechart-1.0.19': {'precision': 0.5338345864661654,
   'recall': 0.6513761467889908,
   'f1': 0.5867768595041323,
   'accuracy': 0.9619627234689996},
  'jruby-1.4.0': {'precision': 0.8401084010840109,
   'rec

In [5]:
import pandas as pd

In [6]:
flat_data = []

for config_key, projects_dict in stored_result.items():
  # Parse out individual hyperparameters from the config key string if desired
  parts = config_key.split('_')
  smote_val = parts[1]
  vote_val = parts[3]
  ratio_val = parts[5]

  for project_name, metrics in projects_dict.items():
    flat_data.append({
        'Configuration': config_key,
        'SMOTE': float(smote_val),
        'Vote_Threshold': int(vote_val),
        'Ratio': float(ratio_val),
        'Project': project_name,
        'Precision': metrics['precision'],
        'Recall': metrics['recall'],
        'F1-Score': metrics['f1'],
        'Accuracy': metrics['accuracy'],
    })

# Convert to DataFrame
df_detailed = pd.DataFrame(flat_data)

# Display the first few rows
df_detailed

,Configuration,SMOTE,Vote_Threshold,Ratio,Project,Precision,Recall,F1-Score,Accuracy
0,smote_0.5_vote_0_ratio_0.1,0.5,0,0.1,apache-jmeter-2.10,0.693333,0.654088,0.673139,0.952526
1,smote_0.5_vote_0_ratio_0.1,0.5,0,0.1,argouml,0.786903,0.616522,0.691370,0.889122
2,smote_0.5_vote_0_ratio_0.1,0.5,0,0.1,columba-1.4-src,0.718750,0.705521,0.712074,0.978517
3,smote_0.5_vote_0_ratio_0.1,0.5,0,0.1,hibernate-distribution-3.3.2.GA,0.824405,0.647196,0.725131,0.918919
4,smote_0.5_vote_0_ratio_0.1,0.5,0,0.1,jEdit-4.2,0.595745,0.238298,0.340426,0.954402
...,...,...,...,...,...,...,...,...,...
67,smote_1.0_vote_4_ratio_0.1,1.0,4,0.1,hibernate-distribution-3.3.2.GA,0.666667,0.747664,0.704846,0.896525
68,smote_1.0_vote_4_ratio_0.1,1.0,4,0.1,jEdit-4.2,0.466368,0.442553,0.454148,0.947468
69,smote_1.0_vote_4_ratio_0.1,1.0,4,0.1,jfreechart-1.0.19,0.369458,0.688073,0.480769,0.938380
70,smote_1.0_vote_4_ratio_0.1,1.0,4,0.1,jruby-1.4.0,0.640429,0.774892,0.701273,0.920199


In [ ]:
best_row = df_detailed.loc[df_detailed["F1-Score"].idxmax()]
print(best_row)

Configuration     smote_0.75_vote_2_ratio_0.1
SMOTE                                    0.75
Vote_Threshold                              2
Ratio                                     0.1
Project                           jruby-1.4.0
Precision                             0.74375
Recall                               0.772727
F1-Score                             0.757962
Accuracy                             0.940345
Name: 38, dtype: object


##Configuration Summary

In [7]:
# Group by Configuration and calculate the average for each metric
df_grouped = (
    df_detailed.groupby("Configuration")[
        ["Precision", "Recall", "F1-Score", "Accuracy"]
    ]
    .mean()
    .reset_index()
)


df_grouped = df_grouped.sort_values(by="F1-Score", ascending=False)


display(df_grouped)

,Configuration,Precision,Recall,F1-Score,Accuracy
4,smote_0.75_vote_2_ratio_0.1,0.619603,0.671136,0.636297,0.939969
0,smote_0.5_vote_0_ratio_0.1,0.704809,0.572890,0.621029,0.945321
5,smote_0.75_vote_4_ratio_0.1,0.698907,0.571440,0.618798,0.944332
3,smote_0.75_vote_0_ratio_0.1,0.526857,0.747887,0.610535,0.926438
8,smote_1.0_vote_4_ratio_0.1,0.530497,0.690215,0.592429,0.926951
1,smote_0.5_vote_2_ratio_0.1,0.771873,0.477214,0.578642,0.944352
7,smote_1.0_vote_2_ratio_0.1,0.436799,0.781370,0.550948,0.902276
2,smote_0.5_vote_4_ratio_0.1,0.827983,0.375848,0.503924,0.939413
6,smote_1.0_vote_0_ratio_0.1,0.324125,0.850141,0.451872,0.822629


In [8]:
df_grouped.to_csv("configurationSummary.csv",index=False)

In [2]:
stored_result={'smote_0.5_vote_0_ratio_0.1': {'apache-jmeter-2.10': {'precision': 0.6933333333333334,
   'recall': 0.6540880503144654,
   'f1': 0.6731391585760518,
   'accuracy': 0.9525264394829612},
  'argouml': {'precision': 0.7869034406215316,
   'recall': 0.6165217391304347,
   'f1': 0.6913700633837152,
   'accuracy': 0.8891224382553863},
  'columba-1.4-src': {'precision': 0.71875,
   'recall': 0.7055214723926381,
   'f1': 0.7120743034055728,
   'accuracy': 0.9785169785169785},
  'hibernate-distribution-3.3.2.GA': {'precision': 0.8244047619047619,
   'recall': 0.647196261682243,
   'f1': 0.725130890052356,
   'accuracy': 0.918918918918919},
  'jEdit-4.2': {'precision': 0.5957446808510638,
   'recall': 0.23829787234042554,
   'f1': 0.3404255319148936,
   'accuracy': 0.9544021853330532},
  'jfreechart-1.0.19': {'precision': 0.5338345864661654,
   'recall': 0.6513761467889908,
   'f1': 0.5867768595041323,
   'accuracy': 0.9619627234689996},
  'jruby-1.4.0': {'precision': 0.8401084010840109,
   'recall': 0.670995670995671,
   'f1': 0.7460890493381468,
   'accuracy': 0.9447933019361591},
  'sql12': {'precision': 0.6453900709219859,
   'recall': 0.3991228070175439,
   'f1': 0.4932249322493225,
   'accuracy': 0.9623211767076365}},
 'smote_0.5_vote_2_ratio_0.1': {'apache-jmeter-2.10': {'precision': 0.764,
   'recall': 0.60062893081761,
   'f1': 0.6725352112676056,
   'accuracy': 0.9562867215041128},
  'argouml': {'precision': 0.8135135135135135,
   'recall': 0.5234782608695652,
   'f1': 0.6370370370370371,
   'accuracy': 0.8798388509371168},
  'columba-1.4-src': {'precision': 0.7798165137614679,
   'recall': 0.5214723926380368,
   'f1': 0.625,
   'accuracy': 0.9764379764379765},
  'hibernate-distribution-3.3.2.GA': {'precision': 0.875,
   'recall': 0.5887850467289719,
   'f1': 0.7039106145251397,
   'accuracy': 0.9181467181467181},
  'jEdit-4.2': {'precision': 0.6724137931034483,
   'recall': 0.16595744680851063,
   'f1': 0.26621160409556316,
   'accuracy': 0.9548224416894305},
  'jfreechart-1.0.19': {'precision': 0.6590909090909091,
   'recall': 0.5321100917431193,
   'f1': 0.5888324873096447,
   'accuracy': 0.9691898060098897},
  'jruby-1.4.0': {'precision': 0.8732876712328768,
   'recall': 0.551948051948052,
   'f1': 0.6763925729442971,
   'accuracy': 0.9361590790162219},
  'sql12': {'precision': 0.7378640776699029,
   'recall': 0.3333333333333333,
   'f1': 0.459214501510574,
   'accuracy': 0.9639331049768285}},
 'smote_0.5_vote_4_ratio_0.1': {'apache-jmeter-2.10': {'precision': 0.841025641025641,
   'recall': 0.5157232704402516,
   'f1': 0.6393762183235867,
   'accuracy': 0.9565217391304348},
  'argouml': {'precision': 0.8198347107438017,
   'recall': 0.43130434782608695,
   'f1': 0.5652421652421652,
   'accuracy': 0.8663513750218953},
  'columba-1.4-src': {'precision': 0.788235294117647,
   'recall': 0.4110429447852761,
   'f1': 0.5403225806451613,
   'accuracy': 0.9736659736659736},
  'hibernate-distribution-3.3.2.GA': {'precision': 0.907563025210084,
   'recall': 0.5046728971962616,
   'f1': 0.6486486486486487,
   'accuracy': 0.9096525096525097},
  'jEdit-4.2': {'precision': 0.7941176470588235,
   'recall': 0.1148936170212766,
   'f1': 0.20074349442379183,
   'accuracy': 0.9548224416894305},
  'jfreechart-1.0.19': {'precision': 0.7368421052631579,
   'recall': 0.3853211009174312,
   'f1': 0.5060240963855421,
   'accuracy': 0.9688094332445797},
  'jruby-1.4.0': {'precision': 0.9029126213592233,
   'recall': 0.4025974025974026,
   'f1': 0.5568862275449101,
   'accuracy': 0.9225536368393511},
  'sql12': {'precision': 0.8333333333333334,
   'recall': 0.2412280701754386,
   'f1': 0.3741496598639456,
   'accuracy': 0.9629256498085835}},
 'smote_0.75_vote_0_ratio_0.1': {'apache-jmeter-2.10': {'precision': 0.4696132596685083,
   'recall': 0.8018867924528302,
   'f1': 0.5923344947735192,
   'accuracy': 0.9175088131609871},
  'argouml': {'precision': 0.7160194174757282,
   'recall': 0.7695652173913043,
   'f1': 0.7418273260687342,
   'accuracy': 0.8921001926782274},
  'columba-1.4-src': {'precision': 0.5035971223021583,
   'recall': 0.8588957055214724,
   'f1': 0.6349206349206349,
   'accuracy': 0.9628089628089628},
  'hibernate-distribution-3.3.2.GA': {'precision': 0.6468330134357005,
   'recall': 0.7873831775700935,
   'f1': 0.7102212855637513,
   'accuracy': 0.8938223938223938},
  'jEdit-4.2': {'precision': 0.4844961240310077,
   'recall': 0.5319148936170213,
   'f1': 0.5070993914807302,
   'accuracy': 0.9489388527001471},
  'jfreechart-1.0.19': {'precision': 0.3497942386831276,
   'recall': 0.7798165137614679,
   'f1': 0.48295454545454547,
   'accuracy': 0.9307721567135793},
  'jruby-1.4.0': {'precision': 0.6397415185783522,
   'recall': 0.8571428571428571,
   'f1': 0.7326549491211841,
   'accuracy': 0.9243851386708529},
  'sql12': {'precision': 0.40476190476190477,
   'recall': 0.5964912280701754,
   'f1': 0.48226950354609927,
   'accuracy': 0.9411646181744913}},
 'smote_0.75_vote_2_ratio_0.1': {'apache-jmeter-2.10': {'precision': 0.565947242206235,
   'recall': 0.7421383647798742,
   'f1': 0.6421768707482993,
   'accuracy': 0.9381903642773208},
  'argouml': {'precision': 0.7504638218923934,
   'recall': 0.7034782608695652,
   'f1': 0.7262118491921006,
   'accuracy': 0.8931511648274654},
  'columba-1.4-src': {'precision': 0.6113744075829384,
   'recall': 0.7914110429447853,
   'f1': 0.6898395721925134,
   'accuracy': 0.9732039732039732},
  'hibernate-distribution-3.3.2.GA': {'precision': 0.7397590361445783,
   'recall': 0.7172897196261683,
   'f1': 0.7283511269276394,
   'accuracy': 0.9115830115830116},
  'jEdit-4.2': {'precision': 0.5670731707317073,
   'recall': 0.39574468085106385,
   'f1': 0.46616541353383456,
   'accuracy': 0.955242698045808},
  'jfreechart-1.0.19': {'precision': 0.4406779661016949,
   'recall': 0.7155963302752294,
   'f1': 0.5454545454545454,
   'accuracy': 0.9505515405096995},
  'jruby-1.4.0': {'precision': 0.74375,
   'recall': 0.7727272727272727,
   'f1': 0.7579617834394905,
   'accuracy': 0.9403453689167975},
  'sql12': {'precision': 0.5377777777777778,
   'recall': 0.5307017543859649,
   'f1': 0.5342163355408388,
   'accuracy': 0.9574853919000604}},
 'smote_0.75_vote_4_ratio_0.1': {'apache-jmeter-2.10': {'precision': 0.668693009118541,
   'recall': 0.6918238993710691,
   'f1': 0.6800618238021638,
   'accuracy': 0.9513513513513514},
  'argouml': {'precision': 0.804147465437788,
   'recall': 0.6069565217391304,
   'f1': 0.6917740336967294,
   'accuracy': 0.8910492205289893},
  'columba-1.4-src': {'precision': 0.6815286624203821,
   'recall': 0.656441717791411,
   'f1': 0.66875,
   'accuracy': 0.9755139755139756},
  'hibernate-distribution-3.3.2.GA': {'precision': 0.835820895522388,
   'recall': 0.6542056074766355,
   'f1': 0.7339449541284404,
   'accuracy': 0.9216216216216216},
  'jEdit-4.2': {'precision': 0.64,
   'recall': 0.2723404255319149,
   'f1': 0.382089552238806,
   'accuracy': 0.9565034671149402},
  'jfreechart-1.0.19': {'precision': 0.5234375,
   'recall': 0.6146788990825688,
   'f1': 0.5654008438818565,
   'accuracy': 0.9608216051730696},
  'jruby-1.4.0': {'precision': 0.7837837837837838,
   'recall': 0.6277056277056277,
   'f1': 0.6971153846153846,
   'accuracy': 0.9340659340659341},
  'sql12': {'precision': 0.6538461538461539,
   'recall': 0.4473684210526316,
   'f1': 0.53125,
   'accuracy': 0.9637316139431795}},
 'smote_1.0_vote_0_ratio_0.1': {'apache-jmeter-2.10': {'precision': 0.29718456725756,
   'recall': 0.8962264150943396,
   'f1': 0.4463586530931872,
   'accuracy': 0.8338425381903642},
  'argouml': {'precision': 0.591922845087402,
   'recall': 0.8539130434782609,
   'f1': 0.6991812032751868,
   'accuracy': 0.8519880889823086},
  'columba-1.4-src': {'precision': 0.26763110307414106,
   'recall': 0.9079754601226994,
   'f1': 0.4134078212290503,
   'accuracy': 0.9029799029799029},
  'hibernate-distribution-3.3.2.GA': {'precision': 0.4622991347342398,
   'recall': 0.8738317757009346,
   'f1': 0.6046887631366209,
   'accuracy': 0.8111969111969112},
  'jEdit-4.2': {'precision': 0.08011869436201781,
   'recall': 0.6893617021276596,
   'f1': 0.1435533894550288,
   'accuracy': 0.5938222315612524},
  'jfreechart-1.0.19': {'precision': 0.2396088019559902,
   'recall': 0.8990825688073395,
   'f1': 0.3783783783783784,
   'accuracy': 0.8775199695701787},
  'jruby-1.4.0': {'precision': 0.4446742502585315,
   'recall': 0.9307359307359307,
   'f1': 0.6018194541637508,
   'accuracy': 0.8511250654107797},
  'sql12': {'precision': 0.20955882352941177,
   'recall': 0.75,
   'f1': 0.3275862068965517,
   'accuracy': 0.8585532943784001}},
 'smote_1.0_vote_2_ratio_0.1': {'apache-jmeter-2.10': {'precision': 0.3765347885402456,
   'recall': 0.8679245283018868,
   'f1': 0.5252140818268316,
   'accuracy': 0.8827262044653349},
  'argouml': {'precision': 0.6595135908440629,
   'recall': 0.8017391304347826,
   'f1': 0.7237048665620094,
   'accuracy': 0.8766859344894027},
  'columba-1.4-src': {'precision': 0.3753351206434316,
   'recall': 0.8588957055214724,
   'f1': 0.5223880597014925,
   'accuracy': 0.9408639408639409},
  'hibernate-distribution-3.3.2.GA': {'precision': 0.5544871794871795,
   'recall': 0.8084112149532711,
   'f1': 0.6577946768060836,
   'accuracy': 0.861003861003861},
  'jEdit-4.2': {'precision': 0.3935860058309038,
   'recall': 0.574468085106383,
   'f1': 0.4671280276816609,
   'accuracy': 0.935280521117882},
  'jfreechart-1.0.19': {'precision': 0.292358803986711,
   'recall': 0.8073394495412844,
   'f1': 0.4292682926829268,
   'accuracy': 0.9109927729174591},
  'jruby-1.4.0': {'precision': 0.5473965287049399,
   'recall': 0.8874458874458875,
   'f1': 0.6771263418662262,
   'accuracy': 0.8976975405546834},
  'sql12': {'precision': 0.29518072289156627,
   'recall': 0.6447368421052632,
   'f1': 0.4049586776859504,
   'accuracy': 0.9129558734636308}},
 'smote_1.0_vote_4_ratio_0.1': {'apache-jmeter-2.10': {'precision': 0.4847908745247148,
   'recall': 0.8018867924528302,
   'f1': 0.6042654028436019,
   'accuracy': 0.9215041128084607},
  'argouml': {'precision': 0.7244986922406277,
   'recall': 0.7226086956521739,
   'f1': 0.7235524597300828,
   'accuracy': 0.8887721142056402},
  'columba-1.4-src': {'precision': 0.49049429657794674,
   'recall': 0.7914110429447853,
   'f1': 0.6056338028169014,
   'accuracy': 0.9611919611919612},
  'hibernate-distribution-3.3.2.GA': {'precision': 0.6666666666666666,
   'recall': 0.7476635514018691,
   'f1': 0.7048458149779736,
   'accuracy': 0.8965250965250965},
  'jEdit-4.2': {'precision': 0.4663677130044843,
   'recall': 0.4425531914893617,
   'f1': 0.45414847161572053,
   'accuracy': 0.9474679554528262},
  'jfreechart-1.0.19': {'precision': 0.3694581280788177,
   'recall': 0.6880733944954128,
   'f1': 0.4807692307692308,
   'accuracy': 0.9383796120197794},
  'jruby-1.4.0': {'precision': 0.6404293381037567,
   'recall': 0.7748917748917749,
   'f1': 0.7012732615083251,
   'accuracy': 0.9201988487702774},
  'sql12': {'precision': 0.4012738853503185,
   'recall': 0.5526315789473685,
   'f1': 0.46494464944649444,
   'accuracy': 0.9415676002417892}}}

In [4]:
from google.colab import files
import pandas as pd


def export_and_download_csv(df, file_name):
  """Exports a DataFrame to CSV and downloads it locally in Google Colab.

  Parameters:
  df (pd.DataFrame): The DataFrame to export.
  file_name (str): The desired file name (e.g., 'grid_search_summary.csv').
  """
  # Ensure the filename ends with .csv
  if not file_name.endswith('.csv'):
    file_name += '.csv'

  # Save DataFrame to CSV without the index column
  df.to_csv(file_name, index=False)
  print(f"File '{file_name}' saved successfully. Downloading...")

  # Trigger browser download in Colab
  files.download(file_name)


# Example usage:
# export_and_download_csv(df_grouped, 'best_satd_grid_search_results.csv')
export_and_download_csv(df_detailed, 'best_satd_grid_search_results.csv')

File 'best_satd_grid_search_results.csv' saved successfully. Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>